In [ ]:
import pandas as pd
hesa = pd.read_csv("/Users/francosebastiani/Documents/GitHub/Economic_Observatory/data/raw_data/facilities/hesa_campus_locations.csv")

print(f"Shape: {hesa.shape}")
print(f"Columns: {hesa.columns.tolist()}")
print(f"\nCountry breakdown:")
print(hesa["CampusCountry"].value_counts())
print(f"\nEngland campuses: {(hesa['CampusCountry'] == 'England').sum()}")
print(f"\nCampuses with coordinates: {hesa['CampusLatitude'].notna().sum()}")

In [ ]:
import geopandas as gpd
from shapely.geometry import Point

# filter to England only and drop missing coordinates
eng = hesa[
    (hesa["CampusCountry"] == "England") &
    hesa["CampusLatitude"].notna() &
    hesa["CampusLongitude"].notna()
].copy()

print(f"England campuses with coordinates: {len(eng)}")

# convert to GeoDataFrame
gdf_campuses = gpd.GeoDataFrame(
    eng,
    geometry=gpd.points_from_xy(eng["CampusLongitude"], eng["CampusLatitude"]),
    crs="EPSG:4326"
)

# load LAD boundaries
lad = gpd.read_file("/Users/francosebastiani/Documents/GitHub/Economic_Observatory/data/raw_data/boundaries/UK_Local_Authority_Districts_December_2023_Boundaries_UK_BGC_2537431731774104276.geojson")
lad = lad[lad["LAD23CD"].str.startswith("E")].copy()
lad = lad.to_crs("EPSG:4326")

# spatial join
joined = gpd.sjoin(gdf_campuses, lad[["LAD23CD", "LAD23NM", "geometry"]], how="left", predicate="within")

# boundary fallback
unmatched = joined[joined["LAD23CD"].isna()].drop(columns=["index_right", "LAD23CD", "LAD23NM"])
if len(unmatched) > 0:
    nearest = gpd.sjoin_nearest(unmatched, lad[["LAD23CD", "LAD23NM", "geometry"]], how="left")
    joined = pd.concat([joined[joined["LAD23CD"].notna()], nearest], ignore_index=True)

print(f"Matched: {joined['LAD23CD'].notna().sum()} of {len(eng)}")

# count distinct providers per LAD
lad_uni = (
    joined.groupby("LAD23CD")["UKPRN"]
    .nunique()
    .reset_index()
    .rename(columns={"UKPRN": "n_universities_lad"})
)

# ensure all 296 LADs present
all_lads = lad[["LAD23CD", "LAD23NM"]]
lad_uni = all_lads.merge(lad_uni, on="LAD23CD", how="left").fillna(0)
lad_uni["n_universities_lad"] = lad_uni["n_universities_lad"].astype(int)

print(f"\nLADs with at least 1 HE provider: {(lad_uni['n_universities_lad'] > 0).sum()}")
print(f"LADs with zero HE providers: {(lad_uni['n_universities_lad'] == 0).sum()}")
print(f"\nTop 10 LADs by HE providers:")
print(lad_uni.nlargest(10, "n_universities_lad")[["LAD23NM", "n_universities_lad"]].to_string())

In [ ]:
print(f"Boundary fallback cases: {len(unmatched)}")